Precision & Recall


In [14]:
from PIL import Image, ImageDraw
from collections import Counter
import cv2
import numpy as np
from ultralytics import YOLO
import os
import supervision as sv

image_path = r"C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\test\images\0000154_00001_d_0000001_jpg.rf.0c05a4605a95233b966a5e07690cfe88.jpg"
labels_path = r"C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\test\labels\0000154_00001_d_0000001_jpg.rf.0c05a4605a95233b966a5e07690cfe88.txt"
model = YOLO(r"C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\runs\segment\train3\weights\best.pt")

default_values = {0:0,  # Car
                  1:0,  # Motorcycle
                  2:0,  # Person
                  3:0,  # Truck
                  4:0}  # Van

########################################### Reading from ground truth

# Open the file and read line by line
with open(labels_path, 'r') as file:
    for line in file:
        # Split the line into words and take the first word
        words = line.split()
        if words:  # Check if the line is not empty
            first_word = words[0]

            # Try to convert the first word to a number
            try:
                number = int(first_word)  # Change to `int` if you expect integers only
                default_values[number] += 1
            except ValueError:
                # If it's not a number, skip this line
                continue

############################################ Reading from custom YOLO model
# Define the path to the image directory
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Do an inference with a mode
results = model.predict(source=image_path)
# Get the detections that were made by the model
detections = sv.Detections.from_ultralytics(results[0])
numbers = detections.class_id

# Count occurrences
original_dict = Counter(default_values)
count_dict = Counter(numbers)
result = original_dict - count_dict  # Subtracting the second Counter from the first
print(result) 
print(original_dict)
print(count_dict)
predicted = count_dict
actual = original_dict
precision = {}
recall = {}

# Iterate through all unique keys in both predicted and actual counters
classes = set(predicted.keys()).union(set(actual.keys()))
for cls in classes:
    TP = min(predicted.get(cls, 0), actual.get(cls, 0))  # True Positives
    FP = max(predicted.get(cls, 0) - actual.get(cls, 0), 0)  # False Positives
    FN = max(actual.get(cls, 0) - predicted.get(cls, 0), 0)  # False Negatives

    # Calculate Precision and Recall for the current class
    if TP + FP > 0:
        precision[cls] = TP / (TP + FP)
    else:
        precision[cls] = 0  # No positives predicted for this class

    if TP + FN > 0:
        recall[cls] = TP / (TP + FN)
    else:
        recall[cls] = 0  # No actual positives for this class

# Print Precision and Recall for each class
for cls in classes:
    print(f"Class {cls}: Precision = {precision[cls]:.2f}, Recall = {recall[cls]:.2f}")


image 1/1 C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\test\images\0000154_00001_d_0000001_jpg.rf.0c05a4605a95233b966a5e07690cfe88.jpg: 384x640 11 Cars, 1 Motorcycle, 6 Persons, 1 Van, 47.8ms
Speed: 3.0ms preprocess, 47.8ms inference, 7.5ms postprocess per image at shape (1, 3, 384, 640)
Counter({2: 9, 1: 7, 4: 2})
Counter({2: 15, 0: 9, 1: 8, 4: 3, 3: 0})
Counter({0: 11, 2: 6, 4: 1, 1: 1})
Class 0: Precision = 0.82, Recall = 1.00
Class 1: Precision = 1.00, Recall = 0.12
Class 2: Precision = 1.00, Recall = 0.40
Class 3: Precision = 0.00, Recall = 0.00
Class 4: Precision = 1.00, Recall = 0.33


In [54]:
from collections import Counter
from PIL import Image, ImageDraw
from collections import Counter
import cv2
import numpy as np
from ultralytics import YOLO
import os
import supervision as sv


def get_labelpath(labels_path):
    filename = labels_path.split("\\")[-1][:-4]
    tmp = labels_path.split("\\")[0:8]
    tmp.extend(["labels", f"{filename}.txt"])
    sep = '/'
    return str(f"{sep.join(tmp)}") 

image_path = r"C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\test\images"
model = YOLO(r"C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\runs\segment\train3\weights\best.pt")

default_values = {0:0,  # Car
                  1:0,  # Motorcycle
                  2:0,  # Person
                  3:0,  # Truck
                  4:0}  # Van

########################################### Reading from ground truth
for filename in os.listdir(image_path):
    # Check if the file is an image (you can adjust this check based on your image formats)
    if filename.endswith(('.jpg', '.jpeg', '.png')):
        image_paths = os.path.join(image_path, filename)
        labels_path = get_labelpath(image_paths)

        # Open the file and read line by line
        with open(labels_path, 'r') as file:
            for line in file:
                # Split the line into words and take the first word
                words = line.split()
                if words:  # Check if the line is not empty
                    first_word = words[0]

                    # Try to convert the first word to a number
                    try:
                        number = int(first_word)  # Change to `int` if you expect integers only
                        default_values[number] += 1
                    except ValueError:
                        # If it's not a number, skip this line
                        continue

        ############################################ Reading from custom YOLO model
        # Define the path to the image directory
        image = cv2.imread(image_paths)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Do an inference with a mode
        results = model.predict(source=image_paths)
        # Get the detections that were made by the model
        detections = sv.Detections.from_ultralytics(results[0])
        numbers = detections.class_id

        # Count occurrences
        original_dict = Counter(default_values)
        count_dict = Counter(numbers)
        
        predicted = count_dict 
        actual = original_dict
        precision = {}
        recall = {}
        print(predicted)
        print(actual)
        # Initialize cumulative sums for Precision and Recall across all classes
        cumulative_precision = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}
        cumulative_recall = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}

        # Initialize a counter for the number of updates (dictionaries processed)
        num_updates = 0


        # Function to update running average for precision and recall
        def update_running_average(predicted, actual):
            global cumulative_precision, cumulative_recall, num_updates

            # Calculate TP, FP, FN for each class
            for cls in cumulative_precision:
                TP = min(predicted.get(cls, 0), actual.get(cls, 0))  # True Positives
                FP = max(predicted.get(cls, 0) - actual.get(cls, 0), 0)  # False Positives
                FN = max(actual.get(cls, 0) - predicted.get(cls, 0), 0)  # False Negatives

                # Update cumulative sums for precision and recall
                if TP + FP > 0:
                    cumulative_precision[cls] += TP / (TP + FP)
                if TP + FN > 0:
                    cumulative_recall[cls] += TP / (TP + FN)

            # Increment the number of updates
            num_updates += 1

            # Calculate the average precision and recall for each class
            average_precision = {cls: cumulative_precision[cls] / num_updates for cls in cumulative_precision}
            average_recall = {cls: cumulative_recall[cls] / num_updates for cls in cumulative_recall}

            return average_precision, average_recall

        # Update running average with the initial dictionaries
        avg_precision, avg_recall = update_running_average(predicted, actual)

# Print the average precision and recall after first update
print("After first update:")
for cls in avg_precision:
    print(f"Class {cls}: Average Precision = {avg_precision[cls]:.2f}, Average Recall = {avg_recall[cls]:.2f}")





image 1/1 C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\test\images\0000001_05999_d_0000011_patch_0_1_jpg.rf.ef5295758cc30034bc6fa203dfd6d930.jpg: 384x640 2 Cars, 6 Persons, 30.8ms
Speed: 2.0ms preprocess, 30.8ms inference, 19.0ms postprocess per image at shape (1, 3, 384, 640)
Counter({2: 6, 0: 2})
Counter({2: 13, 1: 6, 0: 3, 3: 0, 4: 0})

image 1/1 C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\test\images\0000026_00000_d_0000024_patch_0_0_jpg.rf.64b7a931e03c006d53875830c1471612.jpg: 384x640 1 Car, 52.6ms
Speed: 0.0ms preprocess, 52.6ms inference, 4.0ms postprocess per image at shape (1, 3, 384, 640)
Counter({0: 1})
Counter({2: 13, 1: 6, 0: 4, 3: 0, 4: 0})

image 1/1 C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\test\images\0000026_01500_d_0000027_patch_0_0_jpg.rf.b870586c98dcfdb3dc71fd9b8b0fe7ed.jpg: 384x640 5 Cars, 2 Persons, 57.2ms
Speed: 0.0ms preprocess, 57.2ms inference, 5.0ms postprocess 

KeyboardInterrupt: 

In [32]:
filename = labels_path.split("\\")[-1][:-4]
tmp = labels_path.split("\\")[0:7].append(["images", f"{filename}.jpg"])
sep = '\\'
str(f"{sep.join(tmp)}")

TypeError: can only join an iterable

In [47]:
filename = labels_path.split("\\")[-1][:-4] # , f"{filename}.jpg"]
tmp = labels_path.split("\\")[0:7]
tmp.extend(["images", f"{filename}.jpg"])
sep = '\\'
str(f"{sep.join(tmp)}")


In [49]:
sep = '\\'
str(f"{sep.join(tmp)}")

'C:\\Users\\Spacelab3\\Desktop\\envs\\Segmentation\\Object Segmentation.v3i.yolov8\\images\\0000154_00001_d_0000001_jpg.rf.0c05a4605a95233b966a5e07690cfe88.jpg'